<a href="https://colab.research.google.com/github/krushnajayale/boli/blob/main/language_translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [116]:
!pip install streamlit transformers torch sentencepiece pyngrok

In [117]:
%%writefile genz_slang.json
{
  "bruh": "bro",
  "lowkey": "slightly",
  "mid": "average",
  "sus": "suspicious",
  "fr": "for real",
  "ngl": "not going to lie",
  "slaps": "is very good",
  "cap": "lie",
  "no": "no",
  "lit": "exciting"
}

Overwriting genz_slang.json


In [118]:
%%writefile morse_dict.py
MORSE_CODE_DICT = {
    '.-': 'A', '-...': 'B', '-.-.': 'C', '-..': 'D',
    '.': 'E', '..-.': 'F', '--.': 'G', '....': 'H',
    '..': 'I', '.---': 'J', '-.-': 'K', '.-..': 'L',
    '--': 'M', '-.': 'N', '---': 'O', '.--.': 'P',
    '--.-': 'Q', '.-.': 'R', '...': 'S', '-': 'T',
    '..-': 'U', '...-': 'V', '.--': 'W', '-..-': 'X',
    '-.--': 'Y', '--..': 'Z',
    '-----': '0', '.----': '1', '..---': '2',
    '...--': '3', '....-': '4', '.....': '5',
    '-....': '6', '--...': '7', '---..': '8',
    '----.': '9'
}


Overwriting morse_dict.py


In [119]:
%%writefile app.py
import streamlit as st
import json
from transformers import MarianMTModel, MarianTokenizer
from morse_dict import MORSE_CODE_DICT
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # consistent detection

# ---------------- Supported Languages ----------------
SUPPORTED_LANGS = {"en", "hi", "mr", "fr", "de", "es"}

# ---------------- Language Detection ----------------
def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

# ---------------- Load Gen-Z Dictionary ----------------
with open("genz_slang.json", "r") as f:
    GENZ_DICT = json.load(f)

# ---------------- Morse Utilities ----------------
def is_morse(text):
    return all(c in ".-/ " for c in text)

def decode_morse(morse_text):
    words = morse_text.split(" / ")
    decoded_words = []
    for word in words:
        letters = word.split()
        decoded_words.append(''.join(MORSE_CODE_DICT.get(l, '') for l in letters))
    return " ".join(decoded_words)

# ---------------- Gen-Z Normalization ----------------
def normalize_genz(text):
    words = text.lower().split()
    return " ".join(GENZ_DICT.get(word, word) for word in words)

# ---------------- Load Translation Model ----------------
@st.cache_resource
def load_model(src, tgt):
    model_name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    return tokenizer, model

# ---------------- Direct Translation ----------------
def translate_direct(text, src, tgt):
    tokenizer, model = load_model(src, tgt)
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    output = model.generate(**inputs)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# ---------------- Smart Translation (Pivot Logic) ----------------
def smart_translate(text, detected_lang, target_lang):

    # Unsupported language fallback
    if detected_lang not in SUPPORTED_LANGS:
        st.warning(
            f"Detected language '{detected_lang}' is not supported. Treating input as English."
        )
        detected_lang = "en"

    # English → Target
    if detected_lang == "en":
        return translate_direct(text, "en", target_lang)

    # Non-English → English → Target
    to_english = translate_direct(text, detected_lang, "en")
    final_translation = translate_direct(to_english, "en", target_lang)
    return final_translation

# ---------------- Streamlit UI ----------------
st.title("🌐 Gen-Z & Morse Code Translator")

user_text = st.text_area("Enter Gen-Z / Morse / Normal text")

language_map = {
    "Hindi 🇮🇳": "hi",
    "Marathi 🇮🇳": "mr",
    "French 🇫🇷": "fr",
    "German 🇩🇪": "de"
}

selected_language = st.selectbox(
    "🌍 Select Target Language",
    list(language_map.keys())
)

target_lang = language_map[selected_language]

if st.button("🚀 Translate"):

    if user_text.strip() == "":
        st.warning("Please enter some text")
        st.stop()

    # 1️⃣ Morse Decode
    if is_morse(user_text):
        decoded_text = decode_morse(user_text)
        st.subheader("🔤 Morse Decoded Text")
        st.write(decoded_text)
    else:
        decoded_text = user_text

    # 2️⃣ Safe Language Detection
    if len(decoded_text.split()) < 3:
        detected_lang = "en"
    else:
        detected_lang = detect_language(decoded_text)

    st.subheader("🌍 Detected Language")
    st.write(detected_lang)

    # 3️⃣ Gen-Z Normalize (English only)
    if detected_lang == "en":
        normalized_text = normalize_genz(decoded_text)
        st.subheader("🧠 Gen-Z Normalized Text")
        st.write(normalized_text)
    else:
        normalized_text = decoded_text

    # 4️⃣ Smart Translation
    translated = smart_translate(normalized_text, detected_lang, target_lang)

    st.subheader("✅ Final Translation")
    st.success(translated)


Overwriting app.py


In [120]:
!ngrok config add-authtoken 37T04ZcKwd4sYFMCXZtaOjFFOOY_31t8dTZk8nkR7UppVF2Ka


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [121]:
!streamlit run app.py &>/content/logs.txt &


In [122]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(public_url)


NgrokTunnel: "https://acyclic-hortensia-unwatched.ngrok-free.dev" -> "http://localhost:8501"


In [123]:
!pkill ngrok


In [124]:
!streamlit run app.py &>/content/logs.txt &


In [125]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(public_url)


NgrokTunnel: "https://acyclic-hortensia-unwatched.ngrok-free.dev" -> "http://localhost:8501"
